# projet__python_DS
# Le prix de l'immobilier dépend-il de l'accès aux commerces alimentaires ?

*Projet de Python pour la data science (ENSAE, 2026) — [Prénom Nom], [Prénom Nom]*

Notebook 01_nettoyage_bpe

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import geopandas as gpd
from src import config
from src.telechargement import telecharger

In [2]:
colonnes_lues = [
    "DOM", "SDOM", "TYPEQU", "SIRET", "NOMRS", "APET",
    "DEPCOM", "DEP", "REG", "LIBCOM", "CODPOS",
    "LAMBERT_X", "LAMBERT_Y", "EPSG", "LONGITUDE", "LATITUDE",
    "QUALITE_XY", "QUALITE_GEOLOC",
    "DENS3", "DENS7",
]

chemin = telecharger(config.URL_BPE, "BPE25.parquet")
bpe = pd.read_parquet(chemin, columns=colonnes_lues)

BPE25.parquet : téléchargement depuis https://www.insee.fr/fr/statistiques/fichier/8217525/BPE25.parquet ...
BPE25.parquet : 172 Mo téléchargés


In [3]:
bpe = bpe.loc[bpe["SDOM"].isin(["B1", "B2"]) & (bpe["TYPEQU"] != "B103"), :].copy()

In [5]:
bpe_alim = bpe[colonnes_lues].copy()

for col in ["DOM", "SDOM", "TYPEQU", "SIRET", "DEPCOM", "DEP", "REG", "APET", "CODPOS", "DENS7"]:
    bpe_alim[col] = bpe_alim[col].astype("string")

for col in ["LAMBERT_X", "LAMBERT_Y", "LONGITUDE", "LATITUDE"]:
    bpe_alim[col] = pd.to_numeric(bpe_alim[col], errors="coerce")

bpe_alim = bpe_alim.drop_duplicates()
bpe_alim = bpe_alim.dropna(subset=["LONGITUDE", "LATITUDE"]).copy()

In [6]:
DROM = ["971", "972", "973", "974", "976"]
ABSENTS_DVF = ["57", "67", "68"]

bpe_alim = bpe_alim[~bpe_alim["DEP"].isin(DROM + ABSENTS_DVF)].copy()

In [7]:
bpe_geo = gpd.GeoDataFrame(
    bpe_alim,
    geometry=gpd.points_from_xy(bpe_alim["LONGITUDE"], bpe_alim["LATITUDE"]),
    crs="EPSG:4326",
)

bpe_geo = bpe_geo.to_crs("EPSG:2154")

Notebook 02_nettoyage_dvf

In [8]:
from src.densite import charger_grille, ajouter_densite

colonnes_dvf = [
    "id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
    "code_commune", "nom_commune", "code_departement",
    "type_local", "surface_reelle_bati", "surface_terrain",
    "longitude", "latitude",
]

chemin_dvf = telecharger(config.URL_DVF, "dvf_2025.csv.gz")
dvf_brut = pd.read_csv(chemin_dvf, dtype=str, usecols=colonnes_dvf, compression="gzip")

dvf_2025.csv.gz : téléchargement depuis https://www.data.gouv.fr/api/1/datasets/r/b3a0c6d2-2a5a-45a3-81a8-9e860caf5e14 ...
dvf_2025.csv.gz : 116 Mo téléchargés


In [9]:
dvf = dvf_brut[dvf_brut["type_local"].isin(["Maison", "Appartement"])].copy()

dvf = dvf.drop_duplicates(subset=["id_mutation", "type_local", "surface_reelle_bati"])

n_dvf = dvf.groupby("id_mutation").size()

ids_pro = dvf_brut.loc[dvf_brut["type_local"].str.startswith("Local", na=False), "id_mutation"].unique()

ids_un_logement = n_dvf[n_dvf == 1].index

dvf = dvf[dvf["id_mutation"].isin(ids_un_logement) & ~dvf["id_mutation"].isin(ids_pro)].copy()

In [10]:
colonnes_essentielles = [
    "valeur_fonciere", "surface_reelle_bati",
    "code_commune", "longitude", "latitude",
]

dvf = dvf.dropna(subset=colonnes_essentielles).copy()

for col in ["valeur_fonciere", "surface_reelle_bati", "surface_terrain", "longitude", "latitude"]:
    dvf[col] = pd.to_numeric(dvf[col], errors="coerce")

dans_metropole = (dvf["longitude"].between(-5.5, 10) & dvf["latitude"].between(41, 51.5))

dvf = dvf[dans_metropole].copy()
dvf = dvf[dvf["nature_mutation"] == "Vente"].copy()

In [11]:
dvf = dvf[(dvf["valeur_fonciere"] > 0) & (dvf["surface_reelle_bati"] >= 9)].copy()

dvf["prix_m2"] = dvf["valeur_fonciere"] / dvf["surface_reelle_bati"]

PRIX_M2_PLANCHER = 100

dvf = dvf[dvf["prix_m2"] >= PRIX_M2_PLANCHER].copy()

In [12]:
stats_communes = dvf.groupby(["code_commune", "nom_commune"])["prix_m2"].agg(mediane="median", nb_ventes="size")

communes_fiables = stats_communes[stats_communes["nb_ventes"] >= 100]

PRIX_M2_PLAFOND = 10 * communes_fiables["mediane"].max()

dvf = dvf[dvf["prix_m2"] <= PRIX_M2_PLAFOND].copy()

dvf["mediane_commune"] = dvf.groupby("code_commune")["prix_m2"].transform("median")
dvf["ratio"] = dvf["prix_m2"] / dvf["mediane_commune"]

dvf = dvf[dvf["ratio"].between(0.1, 10)].copy()
dvf = dvf.drop(columns=["mediane_commune", "ratio"])

In [13]:
grille = charger_grille(telecharger(config.URL_GRILLE, "grille_densite_2025.xlsx"))

dvf = ajouter_densite(dvf, grille)

grille_densite_2025.xlsx : téléchargement depuis https://www.insee.fr/fr/statistiques/fichier/8571524/fichier_diffusion_2025.xlsx ...
grille_densite_2025.xlsx : 2 Mo téléchargés
Ventes sans densité : 0


In [14]:
dvf_geo = gpd.GeoDataFrame(
    dvf,
    geometry=gpd.points_from_xy(dvf["longitude"], dvf["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:2154")

notebook 03_croisement

In [15]:
from scipy.spatial import cKDTree

bpe = bpe_geo.copy()
dvf = dvf_geo.copy()